# D08 · CAN FD, heartbeats, Cyphal and time-triggered CAN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D08_can_fd_heartbeats_cyphal_and_time_triggered_can.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 55 min

**Skill Hub concepts:** CAN FD data length codes and bit rate switch; CAN heartbeat and node guarding; Cyphal and DroneCAN; Time-triggered CAN

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D08")

## Map: what sits on top of the CAN bit layer

**CAN FD:** up to 64 data bytes; with **BRS** (bit rate switch) the data phase runs faster than arbitration.

| DLC | 0..8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 |
|---|---|---|---|---|---|---|---|---|
| Bytes | 0..8 | 12 | 16 | 20 | 24 | 32 | 48 | 64 |

| Liveness method | Who sends | How a failure is seen |
|---|---|---|
| Heartbeat (CANopen, COB-ID `0x700 + node`, 1 byte NMT state) | every node, periodically | consumer timeout expires (often 1.5 to 3 periods) |
| Node guarding (legacy CANopen) | master polls each node with a remote frame | missing reply; costs bus load, needs remote frames |

| NMT state byte | 0x00 | 0x04 | 0x05 | 0x7F |
|---|---|---|---|---|
| Meaning | boot-up | stopped | operational | pre-operational |

| Protocol | Model | Frame tail byte (last data byte) |
|---|---|---|
| Cyphal (v1) | publish/subscribe on subject IDs, plus services | bit 7 start, bit 6 end, bit 5 toggle (**starts at 1**), bits 4..0 transfer ID |
| DroneCAN (UAVCAN v0) | data types with fixed IDs, used by ArduPilot/PX4 | same layout, toggle **starts at 0** |
| Time-triggered CAN (ISO 11898-4) | a reference message starts each basic cycle; nodes own fixed time windows | no arbitration surprises: deterministic latency |

Data phase time at BRS rate: $t = \dfrac{8n + \text{CRC and overhead}}{\text{data bitrate}}$.

## Task 1: CAN FD lengths

In [ ]:
FD_LEN = [0, 1, 2, 3, 4, 5, 6, 7, 8, 12, 16, 20, 24, 32, 48, 64]

def dlc_for(n_bytes):
    """Smallest DLC whose length holds n_bytes; return (dlc, padded_length)."""
    raise NotImplementedError  # TODO

lab.check("fd", dlc_for)

In [ ]:
%%writefile d08_fd.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d08.h
// Grade it with lab.check("fd", "d08_fd.c"). For C++, name the file .cpp in both places.
#include "d08.h"

int can_fd_dlc_for(size_t n_bytes, size_t *padded_length) {
    (void)n_bytes;
    *padded_length = 0;
    return -1; /* TODO: lengths 0..8, 12, 16, 20, 24, 32, 48, 64 */
}

In [ ]:
# lab.check("fd", "d08_fd.c")  # uncomment after writing the C/C++ cell above

## Task 2: heartbeat consumer
`on_frame(can_id, data, t)` gets every received frame with a timestamp in seconds. `status(t)` returns
`{node: (liveness, nmt_state)}` for the watched nodes, with liveness `"UNKNOWN"` (never heard), `"OK"` or `"LOST"`
(last heartbeat older than the timeout) and `nmt_state` one of `"boot-up"`, `"stopped"`, `"operational"`,
`"pre-operational"` or `None`.

In [ ]:
class HeartbeatMonitor:
    NMT = {0x00: "boot-up", 0x04: "stopped", 0x05: "operational", 0x7F: "pre-operational"}

    def __init__(self, nodes, timeout_s):
        self.nodes, self.timeout = list(nodes), timeout_s
        self.last = {}      # node -> (time, state byte)

    def on_frame(self, can_id, data, t):
        raise NotImplementedError  # TODO

    def status(self, t):
        raise NotImplementedError  # TODO

lab.check("heartbeat", HeartbeatMonitor)

In [ ]:
%%writefile d08_heartbeat.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d08.h
// Grade it with lab.check("heartbeat", "d08_heartbeat.c"). For C++, name the file .cpp in both places.
#include "d08.h"

void hb_init(HeartbeatMonitor *m, const uint8_t *nodes, size_t n_nodes, double timeout_s) {
    m->n_nodes = n_nodes < HB_MAX_NODES ? n_nodes : HB_MAX_NODES;
    m->timeout_s = timeout_s;
    for (size_t i = 0; i < m->n_nodes; ++i) { m->nodes[i] = nodes[i]; m->last_state[i] = -1; m->last_t[i] = 0; }
}

void hb_on_frame(HeartbeatMonitor *m, uint32_t can_id, const uint8_t *data, size_t len, double t) {
    (void)m; (void)can_id; (void)data; (void)len; (void)t; /* TODO: 0x701..0x77F, one data byte, known node */
}

int hb_status(const HeartbeatMonitor *m, uint8_t node, double t, int *nmt_state) {
    (void)m; (void)node; (void)t;
    *nmt_state = -1;
    return HB_UNKNOWN; /* TODO */
}

In [ ]:
# lab.check("heartbeat", "d08_heartbeat.c")  # uncomment after writing the C/C++ cell above

Optional: the same monitor on a live python-can virtual bus (two endpoints in this process).

In [ ]:
import threading, time
import can
bus_tx = can.Bus(interface="virtual", channel="lab-d08")
bus_rx = can.Bus(interface="virtual", channel="lab-d08")
def producer(stop_after):
    t0 = time.time()
    while time.time() - t0 < stop_after:
        bus_tx.send(can.Message(arbitration_id=0x705, data=[0x05], is_extended_id=False))
        time.sleep(0.1)
threading.Thread(target=producer, args=(0.5,), daemon=True).start()
try:
    mon, snap, t0 = HeartbeatMonitor([5, 6], timeout_s=0.3), None, time.time()
    while time.time() - t0 < 1.2:
        msg = bus_rx.recv(timeout=0.05)
        if msg:
            mon.on_frame(msg.arbitration_id, msg.data, msg.timestamp)
        if snap is None and time.time() - t0 > 0.4:
            snap = mon.status(time.time())
    print("at 0.4 s:", snap)
    print("at 1.2 s:", mon.status(time.time()), " (producer stopped at 0.5 s)")
except NotImplementedError:
    print("finish Task 2 first")
bus_tx.shutdown(); bus_rx.shutdown()

## Task 3: tail bytes, Cyphal vs DroneCAN

In [ ]:
def tail_bytes(n_frames, transfer_id, protocol):
    """Tail byte of each frame of one transfer. protocol is "cyphal" or "dronecan"."""
    raise NotImplementedError  # TODO

lab.check("tails", tail_bytes)

In [ ]:
%%writefile d08_tails.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d08.h
// Grade it with lab.check("tails", "d08_tails.c"). For C++, name the file .cpp in both places.
#include "d08.h"

void tail_bytes(size_t n_frames, uint8_t transfer_id, int protocol, uint8_t *out) {
    (void)transfer_id; (void)protocol;
    for (size_t k = 0; k < n_frames; ++k) out[k] = 0; /* TODO: start, end, toggle, transfer id */
}

In [ ]:
# lab.check("tails", "d08_tails.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Remember).** Bytes carried by DLC 13?

**M2 (Apply).** Ignoring CRC and control bits, how many microseconds do 64 data bytes take at a 2 Mbit/s data phase?

**M3 (Apply).** A 21-byte payload is padded to how many bytes?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why can BRS speed up only the data phase?
- a) the data field has its own wires
- b) during arbitration every node must see each bit within one bit time across the whole bus; after arbitration only one transmitter and the receivers' sampling matter
- c) faster arbitration would break the CRC

**K2 (Evaluate).** For a robot arm with six joint boards, compare heartbeat monitoring, node guarding and time-triggered
CAN for detecting a dead joint quickly while keeping bus load and jitter low.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d08.h`).
3. In `roboprep/checks/d08.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()